# g1 Basic Model Evaluation

**Primary author:** Victoria

**Builds on:**
- *05_model_evaluation.ipynb* (Victoria — embedding-loading conventions, `rowwise_cosine` helper, triplet-accuracy patterns)
- *wordplay_ate_breakdown.ipynb* (Victoria — environment auto-detection and figure-directory conventions used under `planning/exploration/`)
- *specs/g1_basic_evaluation.md* (Victoria — this notebook's section outline, outputs, and figure naming)

**Prompt engineering:** Victoria
**AI assistance:** Claude / Claude Code (Anthropic)
**Environment:** Local

Evaluates `g1` as a fine-tuned embedding model using standard data science
practices. Unlike the current NB 05, this notebook progresses from basic
model health (training dynamics, triplet accuracy) through embedding-space
geometry (norms, crowding, effective dimensionality, seen/unseen
stratification) and structural preservation (Spearman on pairwise-cosine
matrices) before returning to the research question (cosine and L2
context effects on the validation set).

Scope is restricted to the **wndef** phrase type throughout — cross-format
(wnex) generalization is a hypothesis-testing question deferred to Stage 6.

Reads artifacts under `custom_embedding_model/models/g1/`,
`data/triplets/`, `data/filtered_split/wn_synset/`, and
`data/embeddings/{g1, g_stock}/`. Writes numerical results to
`outputs/g1_basic_evaluation-results.md` and figures to
`outputs/figures/g1be_*.png`.

Once finalized this notebook will be promoted to
`notebooks/05_model_evaluation.ipynb` and the current NB 05 archived.

## §0 — Setup

Standard imports and environment auto-detection. This notebook lives under
`custom_embedding_model/planning/exploration/`, so the component root is two
directories up and the project root is three directories up. All paths
resolve via `pathlib`. `RANDOM_STATE` is pinned once so the pair sampling in
§3b, the Spearman subsamples in §4, and the stratified pair sampling in §3d
are reproducible across re-runs.

In [ ]:
# === Imports and configuration
import json
import time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy.stats import spearmanr
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# --- Environment auto-detection ---
try:
    IS_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IS_COLAB = False

IS_GREATLAKES = Path("/nfs/turbo").exists()

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Research Project - NLP CCC's/ccc-project")
elif IS_GREATLAKES:
    PROJECT_ROOT = Path.home() / "ccc-project"
else:
    # planning/exploration/ -> custom_embedding_model/ -> ccc-project/
    PROJECT_ROOT = Path("../../..").resolve()

COMPONENT_ROOT = PROJECT_ROOT / "custom_embedding_model"
DATA_DIR       = COMPONENT_ROOT / "data"
WN_DIR         = DATA_DIR / "filtered_split" / "wn_synset"
EMBED_DIR      = DATA_DIR / "embeddings"
TRIPLETS_DIR   = DATA_DIR / "triplets"
MODELS_DIR     = COMPONENT_ROOT / "models"
OUTPUT_DIR     = COMPONENT_ROOT / "outputs"
FIG_DIR        = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

env_label = "Colab" if IS_COLAB else ("Great Lakes" if IS_GREATLAKES else "Local")
print(f"Environment:    {env_label}")
print(f"PROJECT_ROOT:   {PROJECT_ROOT}")
print(f"WN_DIR:         {WN_DIR}")
print(f"EMBED_DIR:      {EMBED_DIR}")
print(f"MODELS_DIR:     {MODELS_DIR}")
print(f"OUTPUT_DIR:     {OUTPUT_DIR}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

NOTEBOOK_T0 = time.time()

In [ ]:
# === Version reporting (Decision 18)
import sys

VERSIONS = {
    "python":     sys.version.split()[0],
    "numpy":      np.__version__,
    "pandas":     pd.__version__,
    "scipy":      scipy.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn":    sns.__version__,
}
for k, v in VERSIONS.items():
    print(f"{k:12s} {v}")

## §1 — Data loading

Three groups of inputs: (a) model artifacts produced by training
(`training_log.json`, `val_loss_results.json`); (b) triplet and clue CSVs
for resolving validation triplets and (clue, definition, answer)
evaluation pairs; (c) embedding arrays and their index / vocabulary files
for both `g_stock` and `g1`.

Per Decision 23, both models now carry full-vocabulary wndef embeddings
(53,930 rows) so every validation triplet and every validation clue
resolves without dropout. For `g_stock` f_clue, the full-dataset array
covers all splits; validation-split rows are extracted by filtering the
shared f_clue index file against `clues_val.csv` so the resulting slice
lines up row-for-row with `g1`'s `f_clue_val.npy` (47,933 rows).

In [ ]:
# === Load model-training artifacts
with open(MODELS_DIR / "g1" / "val_loss_results.json") as fh:
    val_loss_results = json.load(fh)
with open(MODELS_DIR / "g1" / "training_log.json") as fh:
    training_log = json.load(fh)

per_epoch = val_loss_results["per_epoch"]
print(f"val_loss_results: {len(per_epoch)} epochs, hparams margin="
      f"{val_loss_results['margin']}, batch_size={val_loss_results['batch_size']}")
print(f"training_log:     {len(training_log['step_log'])} step entries, "
      f"lr={training_log['hyperparameters']['lr']}, "
      f"epochs={training_log['hyperparameters']['epochs']}")

In [ ]:
# === Load clue and triplet CSVs
t0 = time.time()

clues_wn_filtered = pd.read_csv(
    WN_DIR / "clues_wn_filtered.csv",
    keep_default_na=False, na_values=[""],
)
clues_val = pd.read_csv(
    WN_DIR / "clues_val.csv",
    keep_default_na=False, na_values=[""],
)
vocab_wndef = pd.read_csv(
    WN_DIR / "wndef" / "vocabulary_wndef.csv",
    keep_default_na=False, na_values=[""],
)

train_triplets = pd.read_csv(
    TRIPLETS_DIR / "g1_train.csv",
    keep_default_na=False, na_values=[""],
)
val_triplets = pd.read_csv(
    TRIPLETS_DIR / "g1_val.csv",
    keep_default_na=False, na_values=[""],
)

print(f"clues_wn_filtered: {len(clues_wn_filtered):,} rows "
      f"(splits = {clues_wn_filtered['split'].value_counts().to_dict()})")
print(f"clues_val:         {len(clues_val):,} rows")
print(f"vocabulary_wndef:  {len(vocab_wndef):,} words")
print(f"g1_train.csv:      {len(train_triplets):,} triplets")
print(f"g1_val.csv:        {len(val_triplets):,} triplets")
print(f"Load: {time.time() - t0:.1f}s")

In [ ]:
# === Load embedding arrays and index files
t0 = time.time()

MODEL_NAMES = ["g_stock", "g1"]

# f_common_wndef — full-vocab, 53,930 rows, for both models (Decision 23).
# f_clue_val     — g1 only (validation-split f_clue embeddings).
# f_clue         — g_stock only (full-dataset, covers all splits).
embeddings = {
    ("g1", "f_common_wndef"):     np.load(EMBED_DIR / "g1"      / "f_common_wndef.npy"),
    ("g_stock", "f_common_wndef"):np.load(EMBED_DIR / "g_stock" / "f_common_wndef.npy"),
    ("g1", "f_clue_val"):         np.load(EMBED_DIR / "g1"      / "f_clue_val.npy"),
}
# g_stock full-dataset f_clue: we extract the val-split slice below.
g_stock_f_clue_full = np.load(EMBED_DIR / "g_stock" / "f_clue.npy")

# Index files for f_clue.
g1_f_clue_val_index = pd.read_csv(
    EMBED_DIR / "g1" / "f_clue_val_index.csv",
    keep_default_na=False, na_values=[""],
)
g_stock_f_clue_index = pd.read_csv(
    EMBED_DIR / "g_stock" / "f_clue_index.csv",
    keep_default_na=False, na_values=[""],
)

# --- Shape validation (per CLAUDE.md) ---
assert embeddings[("g1", "f_common_wndef")].shape     == (len(vocab_wndef), 1024)
assert embeddings[("g_stock", "f_common_wndef")].shape== (len(vocab_wndef), 1024)
assert embeddings[("g1", "f_clue_val")].shape         == (len(g1_f_clue_val_index), 1024)
assert g_stock_f_clue_full.shape                       == (len(g_stock_f_clue_index), 1024)
assert embeddings[("g1", "f_clue_val")].shape[0]      == 47_933
assert embeddings[("g1", "f_common_wndef")].shape[0]  == 53_930

print(f"g1/f_common_wndef:      {embeddings[('g1','f_common_wndef')].shape}")
print(f"g_stock/f_common_wndef: {embeddings[('g_stock','f_common_wndef')].shape}")
print(f"g1/f_clue_val:          {embeddings[('g1','f_clue_val')].shape}")
print(f"g_stock/f_clue (full):  {g_stock_f_clue_full.shape}")
print(f"Load + validate: {time.time() - t0:.1f}s")

In [ ]:
# === Extract g_stock f_clue_val aligned to g1 f_clue_val_index
# g_stock f_clue covers all splits (239,406 rows); g1 f_clue_val covers
# only validation (47,933 rows). We build g_stock's validation slice in
# the SAME row order as g1's index so that row i in both models
# corresponds to the same (clue_id, definition) pair.
t0 = time.time()

g_stock_clue_key_to_row = {
    (cid, defn): row
    for cid, defn, row in zip(
        g_stock_f_clue_index["clue_id"],
        g_stock_f_clue_index["definition"],
        g_stock_f_clue_index["row"],
    )
}

g_stock_val_rows = np.array([
    g_stock_clue_key_to_row[(cid, defn)]
    for cid, defn in zip(g1_f_clue_val_index["clue_id"],
                         g1_f_clue_val_index["definition"])
], dtype=np.int64)

embeddings[("g_stock", "f_clue_val")] = g_stock_f_clue_full[g_stock_val_rows]
assert embeddings[("g_stock", "f_clue_val")].shape == embeddings[("g1", "f_clue_val")].shape

# The big full-dataset array is no longer needed once the slice is cached.
del g_stock_f_clue_full

print(f"g_stock/f_clue_val (extracted): {embeddings[('g_stock','f_clue_val')].shape}")
print(f"Row alignment verified against g1/f_clue_val_index.")
print(f"Extract: {time.time() - t0:.1f}s")

In [ ]:
# === Lookup dicts, rowwise_cosine helper, and figure color constants
wndef_word_to_row = dict(zip(vocab_wndef["word"], vocab_wndef["row"]))

# g1 and g_stock f_clue_val share the same index ordering after the alignment
# above, so a single (clue_id, definition) -> row dict serves both.
clue_key_to_row = {
    (cid, defn): row
    for cid, defn, row in zip(
        g1_f_clue_val_index["clue_id"],
        g1_f_clue_val_index["definition"],
        g1_f_clue_val_index["row"],
    )
}

def rowwise_cosine(A, B):
    '''Per-row cosine similarity between two equal-shape (N, D) arrays.'''
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-10)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-10)
    return np.sum(A_norm * B_norm, axis=1)

# FIGURE_STANDARDS.md encoding: model identity -> color; treatment condition
# (T=0 vs T=1) -> outline-only vs filled histogram in the model's full hue;
# metric (cosine vs L2) -> solid fill vs "//" diagonal hatching.
MODEL_COLORS = {"g_stock": "#1f77b4", "g1": "#ff7f0e"}

print(f"wndef_word_to_row: {len(wndef_word_to_row):,} entries")
print(f"clue_key_to_row:   {len(clue_key_to_row):,} entries")

## §2 — Training dynamics and overfitting

How did training and validation loss change over the three epochs? Did the
model keep improving on held-out data, or did it start memorizing? Which
checkpoint generalized best?

Validation loss was computed retroactively from saved epoch checkpoints
per Decision 24 (the original training script did not track it). The
deployed model is epoch 3 — so this section asks both *which* epoch
generalized best and *by how much* the deployed epoch diverged from
that optimum.

In [ ]:
# === Per-epoch training dynamics table
td_rows = []
for rec in per_epoch:
    td_rows.append({
        "epoch":            rec["epoch"],
        "train_loss":       rec["train_loss"],
        "val_loss":         rec["val_loss"],
        "ratio (val/train)": rec["val_loss"] / rec["train_loss"],
        "val_accuracy":     rec["val_accuracy"],
        "val_mean_margin":  rec["val_mean_margin"],
        "val_median_margin":rec["val_median_margin"],
    })
td_df = pd.DataFrame(td_rows)

with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(td_df.to_string(index=False))

# Best-generalizing epoch = epoch with lowest val loss.
best_epoch = int(td_df.loc[td_df["val_loss"].idxmin(), "epoch"])
deployed_epoch = int(td_df["epoch"].max())
print(f"\nBest-generalizing checkpoint: epoch {best_epoch} "
      f"(val loss = {td_df.loc[td_df['epoch']==best_epoch, 'val_loss'].iloc[0]:.4f})")
print(f"Deployed checkpoint:          epoch {deployed_epoch} "
      f"(val loss = {td_df.loc[td_df['epoch']==deployed_epoch, 'val_loss'].iloc[0]:.4f}, "
      f"train loss = {td_df.loc[td_df['epoch']==deployed_epoch, 'train_loss'].iloc[0]:.4f}, "
      f"ratio = {td_df.loc[td_df['epoch']==deployed_epoch, 'ratio (val/train)'].iloc[0]:.1f}x)")

## §3 — Task performance

When given a clue, the correct answer, and a distractor, how often does g1
place the clue embedding closer to the answer than to the distractor? Is
it doing this in both L2 (the metric it was trained on) and cosine (the
metric we use for research)?

### §3a — Triplet evaluation function

A reusable function that, given anchor embeddings, positive embeddings,
and negative embeddings, computes L2 and cosine triplet accuracies,
margins, and the raw per-triplet vectors. Reused in §4d (seen/unseen
stratification).

In [ ]:
# === Reusable triplet evaluation helper
def triplet_eval(A, P, N):
    '''L2 + cosine triplet accuracy and margins for matched (A, P, N) rows.'''
    # Cosine: larger cos(a, p) than cos(a, n) = correct.
    cos_ap = rowwise_cosine(A, P)
    cos_an = rowwise_cosine(A, N)
    cos_margin = cos_ap - cos_an

    # L2: smaller ||a - p|| than ||a - n|| = correct; the "margin" is the
    # negative's distance minus the positive's, so positive = correct.
    l2_ap = np.linalg.norm(A - P, axis=1)
    l2_an = np.linalg.norm(A - N, axis=1)
    l2_margin = l2_an - l2_ap

    return {
        "n":                  len(A),
        "cos_accuracy":       float((cos_margin > 0).mean()),
        "l2_accuracy":        float((l2_margin  > 0).mean()),
        "cos_mean_margin":    float(cos_margin.mean()),
        "cos_median_margin":  float(np.median(cos_margin)),
        "l2_mean_margin":     float(l2_margin.mean()),
        "l2_median_margin":   float(np.median(l2_margin)),
        "_cos_margin":        cos_margin,
        "_l2_margin":         l2_margin,
    }

### §3b — Validation triplet accuracy

Resolve all 46,506 validation triplets against existing embeddings:
anchors from `f_clue_val` via `(clue_id, definition)`, positives and
negatives from `f_common_wndef` via the full-vocab `vocabulary_wndef`
dict. With Decision 23 full-vocab embeddings, resolution is expected to
be 100%.

In [ ]:
# === Resolve val triplet rows against full-vocab wndef
t0 = time.time()

anchor_rows = np.array([
    clue_key_to_row.get((cid, defn), -1)
    for cid, defn in zip(val_triplets["clue_id"], val_triplets["definition"])
])
pos_rows = np.array([
    wndef_word_to_row.get(w, -1) for w in val_triplets["answer_wn"]
])
neg_rows = np.array([
    wndef_word_to_row.get(w, -1) for w in val_triplets["distractor_wn"]
])

n_total = len(val_triplets)
n_miss = {
    "anchor":   int((anchor_rows < 0).sum()),
    "positive": int((pos_rows    < 0).sum()),
    "negative": int((neg_rows    < 0).sum()),
}
print(f"Total triplets:    {n_total:,}")
for role, n in n_miss.items():
    print(f"  {role:9s} misses: {n:,}")

# Under Decision 23 every role must resolve (no val-only vocab gap).
assert all(v == 0 for v in n_miss.values()), (
    f"Unexpected resolution gap: {n_miss}"
)
n_resolved = n_total
print(f"Resolution rate: {n_resolved:,} / {n_total:,} ({n_resolved / n_total:.1%})")
print(f"Resolve: {time.time() - t0:.1f}s")

In [ ]:
# === g1 and g_stock validation triplet stats
def run_triplet_eval(model):
    A = embeddings[(model, "f_clue_val")][anchor_rows]
    P = embeddings[(model, "f_common_wndef")][pos_rows]
    N = embeddings[(model, "f_common_wndef")][neg_rows]
    return triplet_eval(A, P, N)

val_triplet_stats = {model: run_triplet_eval(model) for model in MODEL_NAMES}

rows = []
for model in MODEL_NAMES:
    s = val_triplet_stats[model]
    rows.append({
        "model":              model,
        "n_triplets":         s["n"],
        "l2_accuracy":        s["l2_accuracy"],
        "cos_accuracy":       s["cos_accuracy"],
        "l2_mean_margin":     s["l2_mean_margin"],
        "l2_median_margin":   s["l2_median_margin"],
        "cos_mean_margin":    s["cos_mean_margin"],
        "cos_median_margin":  s["cos_median_margin"],
    })
val_triplet_df = pd.DataFrame(rows)

with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(val_triplet_df.to_string(index=False))

g1_l2  = val_triplet_stats["g1"]["l2_accuracy"]
g1_cos = val_triplet_stats["g1"]["cos_accuracy"]
print(f"\ng1 L2 - cosine accuracy gap: {(g1_l2 - g1_cos)*100:+.2f} pp")
print("If large and positive, the model learned magnitude-based discrimination "
      "invisible to cosine evaluation.")

### §3c — Training triplet accuracy (inferred)

`g1` was trained before Decision 25 (which requires training-split
`f_clue_train` embeddings for each fine-tuned model), so we cannot
compute training triplet accuracy directly. It can still be bounded from
the recorded training loss: with margin = 1.0, the triplet margin loss
is `max(0, ||a - p|| - ||a - n|| + 1.0)`. The epoch-3 training loss of
~0.014 means the *average* hinge violation is 0.014 — so nearly every
training triplet satisfies the L2 margin constraint. L2 training
accuracy is therefore approximately 99% or better.

In [ ]:
# === Inferred training triplet accuracy from training loss
epoch3_train_loss = per_epoch[-1]["train_loss"]
margin = val_loss_results["margin"]
print(f"Epoch-3 training loss: {epoch3_train_loss:.4f} (margin={margin:.1f})")
print(f"Implied L2 training accuracy: approximately "
      f">= {1 - epoch3_train_loss/margin:.1%} (lower bound — actual")
print(f"value depends on the distribution of hinge violations).")
print("\nTraining triplet accuracy cannot be computed exactly without")
print("f_clue_train embeddings (Decision 25 — not yet generated for g1).")

In [ ]:
# === Figure: training dynamics (g1be_training_dynamics.png)
# Spans §2 (loss curves) and §3 (per-epoch task metrics with §3b/§3c
# references overlaid). Per FIGURE_STANDARDS.md: training data = green
# (#2ca02c), validation data = red (#d62728), model identity = blue
# (#1f77b4) for g_stock.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

TRAIN_COLOR   = "#2ca02c"
VAL_COLOR     = "#d62728"
GSTOCK_COLOR  = MODEL_COLORS["g_stock"]

# Reference values pulled from earlier cells (no hardcoded numbers).
gstock_cos_acc    = val_triplet_stats["g_stock"]["cos_accuracy"]
gstock_cos_margin = val_triplet_stats["g_stock"]["cos_mean_margin"]
train_acc_lb      = 1 - epoch3_train_loss / margin

# Panel 1: training vs validation loss.
ax = axes[0]
ax.plot(td_df["epoch"], td_df["train_loss"], "o-", color=TRAIN_COLOR, label="train loss")
ax.plot(td_df["epoch"], td_df["val_loss"],   "o-", color=VAL_COLOR,   label="val loss")
ax.set_xticks([1, 2, 3])
ax.set_xlabel("Epoch")
ax.set_ylabel(r"Loss (triplet margin, $\alpha$ = 1.0)")
ax.set_title("Training vs validation loss")
ax.grid(alpha=0.3)
ax.legend(loc="best")

# Panel 2: validation cosine triplet accuracy. The validation curve is the
# only legend entry; g_stock and inferred-train references are labeled
# directly on their lines so the legend stays uncluttered.
ax = axes[1]
ax.plot(td_df["epoch"], td_df["val_accuracy"], "o-", color=VAL_COLOR,
        label="val accuracy")
ax.axhline(gstock_cos_acc, color=GSTOCK_COLOR, linestyle="--", linewidth=1.5)
ax.axhline(train_acc_lb,   color=TRAIN_COLOR,  linestyle="--", linewidth=1.5)
ax.text(1.05, gstock_cos_acc + 0.035,
        f"g_stock ({gstock_cos_acc*100:.1f}%)",
        color=GSTOCK_COLOR, fontsize=10, va="bottom", ha="left")
ax.text(1.05, train_acc_lb - 0.035,
        f"train ≥ {train_acc_lb*100:.1f}% (inferred)",
        color=TRAIN_COLOR, fontsize=10, va="top", ha="left")
ax.set_xticks([1, 2, 3])
ax.set_xlabel("Epoch")
ax.set_ylabel("Triplet accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Validation cosine triplet accuracy")
ax.grid(alpha=0.3)
ax.legend(loc="lower right")

# Panel 3: validation mean cosine margin. g_stock reference is labeled on
# the line rather than in the legend.
ax = axes[2]
ax.plot(td_df["epoch"], td_df["val_mean_margin"], "o-", color=VAL_COLOR,
        label="val mean margin")
ax.axhline(gstock_cos_margin, color=GSTOCK_COLOR, linestyle="--", linewidth=1.5)
ax.set_xticks([1, 2, 3])
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean cosine margin")
# g_stock cosine margin is negative; widen ylim to show both the reference
# line and the validation curve.
ax.set_ylim(min(gstock_cos_margin, 0) - 0.02, td_df["val_mean_margin"].max() + 0.02)
ax.axhline(0, color="gray", linewidth=1.5)
margin_y_range = (td_df["val_mean_margin"].max() + 0.02) \
                 - (min(gstock_cos_margin, 0) - 0.02)
margin_offset  = margin_y_range * 0.04
ax.text(1.05, gstock_cos_margin + margin_offset,
        f"g_stock ({gstock_cos_margin:.3f})",
        color=GSTOCK_COLOR, fontsize=10, va="bottom", ha="left")
ax.set_title("Validation mean margin")
ax.grid(alpha=0.3)
ax.legend(loc="best")

fig.tight_layout()
fig.savefig(FIG_DIR / "g1be_training_dynamics.png", dpi=300, bbox_inches="tight")
plt.show()


### §3d — Summary table

In [ ]:
# === Summary of task performance
summary_rows = [
    {"Metric":           "L2 accuracy",
     "g_stock val":      val_triplet_stats["g_stock"]["l2_accuracy"],
     "g1 val":           val_triplet_stats["g1"]["l2_accuracy"],
     "g1 train (infer)": f">={1 - epoch3_train_loss/margin:.3f}"},
    {"Metric":           "Cosine accuracy",
     "g_stock val":      val_triplet_stats["g_stock"]["cos_accuracy"],
     "g1 val":           val_triplet_stats["g1"]["cos_accuracy"],
     "g1 train (infer)": "—"},
    {"Metric":           "L2 mean margin",
     "g_stock val":      val_triplet_stats["g_stock"]["l2_mean_margin"],
     "g1 val":           val_triplet_stats["g1"]["l2_mean_margin"],
     "g1 train (infer)": "—"},
    {"Metric":           "Cosine mean margin",
     "g_stock val":      val_triplet_stats["g_stock"]["cos_mean_margin"],
     "g1 val":           val_triplet_stats["g1"]["cos_mean_margin"],
     "g1 train (infer)": "—"},
]
task_summary_df = pd.DataFrame(summary_rows)
with pd.option_context("display.width", 140):
    print(task_summary_df.to_string(index=False))

## §4 — Embedding space geometry

How big are the embedding vectors, and did fine-tuning shrink them? Are
random word pairs more similar to each other under g1 than under g_stock
(crowding)? Is the space still using many dimensions, or has variance
collapsed into a few? Did training reshape the space differently for
vocabulary words it saw during training versus words it never saw?

### §4a — Norm distributions

In [ ]:
# === Norm distributions for wndef and f_clue populations
def norm_stats(vec, label):
    return {
        "Population": label,
        "Mean":   float(vec.mean()),
        "Std":    float(vec.std()),
        "Min":    float(vec.min()),
        "Max":    float(vec.max()),
        "P5":     float(np.percentile(vec, 5)),
        "P95":    float(np.percentile(vec, 95)),
    }

norms = {}
for model in MODEL_NAMES:
    for phrase in ["f_common_wndef", "f_clue_val"]:
        emb = embeddings[(model, phrase)]
        norms[(model, phrase)] = np.linalg.norm(emb, axis=1)

norm_rows = []
for (model, phrase), vec in norms.items():
    pop_label = "wndef vocab (53,930)" if phrase == "f_common_wndef" else "f_clue val (47,933)"
    rec = norm_stats(vec, pop_label)
    rec = {"Population": pop_label, "Model": model, **{k: v for k, v in rec.items() if k != "Population"}}
    norm_rows.append(rec)
norm_df = pd.DataFrame(norm_rows)

with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(norm_df.to_string(index=False))

In [ ]:
# === Figure: norm distributions (g1be_norm_distributions.png)
# L2 norms -> histograms carry diagonal hatching per FIGURE_STANDARDS.md.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

panel_config = [
    (axes[0], "f_common_wndef", "wndef vocabulary norms"),
    (axes[1], "f_clue_val",     "f_clue (val) norms"),
]

for ax, phrase, title in panel_config:
    for model in MODEL_NAMES:
        v = norms[(model, phrase)]
        ax.hist(v, bins=60, alpha=0.5, color=MODEL_COLORS[model],
                hatch="//", edgecolor=MODEL_COLORS[model],
                label=model, density=True)
        ax.axvline(v.mean(), color=MODEL_COLORS[model],
                   linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("L2 norm")
    ax.set_ylabel("Density")
    ax.grid(alpha=0.3)
    ax.legend(loc="best")

fig.tight_layout()
fig.savefig(FIG_DIR / "g1be_norm_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

### §4b — Pairwise cosine among random word pairs

High random-pair cosine means embeddings are crowded together — loss of
angular discriminability. Sampling 50,000 random distinct-row pairs from
each population and using the same pair indices across models makes the
two-model comparison apples-to-apples.

In [ ]:
# === Sample pairs once per population, reuse across models
N_PAIRS = 50_000

def sample_pairs(n_rows, n_pairs, seed):
    rng = np.random.default_rng(seed)
    i = rng.integers(0, n_rows, size=n_pairs)
    j = rng.integers(0, n_rows, size=n_pairs)
    same = i == j
    while same.any():
        j[same] = rng.integers(0, n_rows, size=int(same.sum()))
        same = i == j
    return i, j

pair_sims = {}           # (model, phrase) -> cosine vector
pair_stats_rows = []
for phrase in ["f_common_wndef", "f_clue_val"]:
    n_rows = embeddings[(MODEL_NAMES[0], phrase)].shape[0]
    i_idx, j_idx = sample_pairs(n_rows, N_PAIRS, seed=RANDOM_STATE)
    pop_label = "wndef vocab" if phrase == "f_common_wndef" else "f_clue val"
    for model in MODEL_NAMES:
        emb = embeddings[(model, phrase)]
        sims = rowwise_cosine(emb[i_idx], emb[j_idx])
        pair_sims[(model, phrase)] = sims
        pair_stats_rows.append({
            "Population": pop_label,
            "Model":      model,
            "Mean":       float(sims.mean()),
            "Median":     float(np.median(sims)),
            "Std":        float(sims.std()),
            "P5":         float(np.percentile(sims, 5)),
            "P95":        float(np.percentile(sims, 95)),
        })

pair_df = pd.DataFrame(pair_stats_rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(pair_df.to_string(index=False))

In [ ]:
# === Figure: pairwise cosine (g1be_pairwise_cosine.png)
# Two-panel layout matches g1be_norm_distributions.png so the wndef and
# f_clue populations can be read across figures at the same glance.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

panel_config = [
    (axes[0], "f_common_wndef", "wndef random pairs (N=50,000)"),
    (axes[1], "f_clue_val",     "f_clue val random pairs (N=50,000)"),
]

for ax, phrase, title in panel_config:
    for model in MODEL_NAMES:
        v = pair_sims[(model, phrase)]
        ax.hist(v, bins=60, alpha=0.5, color=MODEL_COLORS[model],
                label=model, density=True)
        ax.axvline(v.mean(), color=MODEL_COLORS[model],
                   linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Cosine similarity")
    ax.set_ylabel("Density")
    ax.grid(alpha=0.3)
    ax.legend(loc="best")

fig.tight_layout()
fig.savefig(FIG_DIR / "g1be_pairwise_cosine.png", dpi=300, bbox_inches="tight")
plt.show()

### §4c — Effective dimensionality

Participation ratio `(Σsᵢ²)² / Σsᵢ⁴` over centered singular values
measures how many dimensions meaningfully contribute to variance. Values
close to the ambient dimensionality (1024) indicate a well-spread space;
small values indicate that variance is concentrated in a few directions.
Cumulative variance curves give a more direct visual read.

In [ ]:
# === SVD + effective dimensionality per embedding matrix
def effective_dim(mat):
    '''Return (singular_values, total_variance, effective_dim, cum_variance).'''
    centered = mat - mat.mean(axis=0, keepdims=True)
    s = np.linalg.svd(centered, full_matrices=False, compute_uv=False)
    s2 = s ** 2
    total = float(s2.sum())
    eff = float((s2.sum() ** 2) / (s2 ** 2).sum())
    cum = np.cumsum(s2) / s2.sum()
    return s, total, eff, cum

eff_data = {}
eff_rows = []
t0 = time.time()
for model in MODEL_NAMES:
    for phrase in ["f_common_wndef", "f_clue_val"]:
        s, total, eff, cum = effective_dim(embeddings[(model, phrase)])
        eff_data[(model, phrase)] = (s, total, eff, cum)
        pop_label = "wndef vocab" if phrase == "f_common_wndef" else "f_clue val"
        eff_rows.append({
            "Population":  pop_label,
            "Model":       model,
            "Total var":   total,
            "Eff. dim":    eff,
            "Top-10 %":    float(cum[9])  * 100,
            "Top-50 %":    float(cum[49]) * 100,
            "Top-100 %":   float(cum[99]) * 100,
        })
eff_df = pd.DataFrame(eff_rows)

with pd.option_context("display.float_format", "{:.2f}".format, "display.width", 140):
    print(eff_df.to_string(index=False))
print(f"\nSVD total runtime: {time.time() - t0:.1f}s")

In [ ]:
# === Figure: cumulative variance (g1be_singular_values.png)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
K_MAX = 200

panel_config = [
    (axes[0], "f_common_wndef", "wndef vocabulary"),
    (axes[1], "f_clue_val",     "f_clue (val)"),
]

for ax, phrase, title in panel_config:
    for model in MODEL_NAMES:
        _, _, _, cum = eff_data[(model, phrase)]
        k = np.arange(1, min(K_MAX, len(cum)) + 1)
        ax.plot(k, cum[:len(k)] * 100, color=MODEL_COLORS[model], label=model)
    ax.set_xlabel("Number of components")
    ax.set_ylabel("Cumulative variance explained (%)")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(loc="best")

fig.tight_layout()
fig.savefig(FIG_DIR / "g1be_singular_values.png", dpi=300, bbox_inches="tight")
plt.show()

### §4d — Seen/unseen vocabulary stratification

Did training reshape the embedding space differently for words the model
saw during training versus words it never encountered? Three word sets:

- `seen_wndef` — `answer_wn ∪ distractor_wn` drawn from `g1_train.csv`.
  These words were directly embedded as wndef phrases during training.
- `seen_fclue_only` — training-split definitions (from `clues_wn_filtered`
  with `split == 'train'`) that are NOT in `seen_wndef`. Seen only via
  f_clue context, never as wndef phrases.
- `unseen` — vocabulary_wndef words in neither set.

If g1 treats seen and unseen words very differently — higher accuracy,
more compression, different norms — that indicates the model's
performance depends on direct optimization of those word vectors rather
than generalized learning.

In [ ]:
# === Build seen/unseen word sets
seen_wndef = set(train_triplets["answer_wn"]) | set(train_triplets["distractor_wn"])
train_clue_defs = set(clues_wn_filtered.loc[
    clues_wn_filtered["split"] == "train", "definition_wn"
])
seen_fclue_only = train_clue_defs - seen_wndef
all_vocab = set(vocab_wndef["word"])
unseen = all_vocab - seen_wndef - seen_fclue_only

print(f"vocabulary_wndef total: {len(all_vocab):,}")
print(f"  seen_wndef:           {len(seen_wndef & all_vocab):,} "
      f"({(len(seen_wndef & all_vocab) / len(all_vocab)):.1%} of wndef)")
print(f"  seen_fclue_only:      {len(seen_fclue_only & all_vocab):,} "
      f"({(len(seen_fclue_only & all_vocab) / len(all_vocab)):.1%} of wndef)")
print(f"  unseen:               {len(unseen):,} "
      f"({(len(unseen) / len(all_vocab)):.1%} of wndef)")

# Restrict each set to vocabulary_wndef (distractors may include words not in
# the wndef subset in principle; restrict to what's indexable).
seen_wndef_in = seen_wndef & all_vocab
seen_fclue_only_in = seen_fclue_only & all_vocab
# Partition check.
assert seen_wndef_in.isdisjoint(seen_fclue_only_in)
assert seen_wndef_in.isdisjoint(unseen)
assert seen_fclue_only_in.isdisjoint(unseen)
assert len(seen_wndef_in) + len(seen_fclue_only_in) + len(unseen) == len(all_vocab)

In [ ]:
# === Norm stratification by word set
def rows_for_set(word_set):
    return np.array([wndef_word_to_row[w] for w in word_set
                     if w in wndef_word_to_row], dtype=np.int64)

set_rows = {
    "seen_wndef":      rows_for_set(seen_wndef_in),
    "seen_fclue_only": rows_for_set(seen_fclue_only_in),
    "unseen":          rows_for_set(unseen),
}

norm_strat_rows = []
mean_norms_by_set = {}   # for the bar chart in §4d figure
for set_name, rows in set_rows.items():
    for model in MODEL_NAMES:
        sub_norms = norms[(model, "f_common_wndef")][rows]
        mean_norms_by_set[(model, set_name)] = float(sub_norms.mean())
        norm_strat_rows.append({
            "Word set": set_name,
            "Model":    model,
            "N":        len(rows),
            "Mean norm":float(sub_norms.mean()),
            "Std":      float(sub_norms.std()),
        })
norm_strat_df = pd.DataFrame(norm_strat_rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(norm_strat_df.to_string(index=False))

In [ ]:
# === Pairwise-cosine stratification (seen_wndef vs unseen)
N_PAIRS_STRAT = 50_000

pair_strat_rows = []
for set_name in ["seen_wndef", "unseen"]:
    rows = set_rows[set_name]
    i_loc, j_loc = sample_pairs(len(rows), N_PAIRS_STRAT, seed=RANDOM_STATE)
    # Translate local indices into vocab rows.
    ri = rows[i_loc]
    rj = rows[j_loc]
    for model in MODEL_NAMES:
        emb = embeddings[(model, "f_common_wndef")]
        sims = rowwise_cosine(emb[ri], emb[rj])
        pair_strat_rows.append({
            "Word set":  set_name,
            "Model":     model,
            "N pairs":   len(sims),
            "Mean cos":  float(sims.mean()),
            "Median":    float(np.median(sims)),
            "Std":       float(sims.std()),
        })
pair_strat_df = pd.DataFrame(pair_strat_rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(pair_strat_df.to_string(index=False))

In [ ]:
# === Triplet-accuracy stratification by (pos in seen_wndef) x (neg in seen_wndef)
pos_seen = np.array([w in seen_wndef_in for w in val_triplets["answer_wn"]])
neg_seen = np.array([w in seen_wndef_in for w in val_triplets["distractor_wn"]])

strata = {
    "both seen_wndef":           pos_seen & neg_seen,
    "both unseen":              ~pos_seen & ~neg_seen,
    "mixed (one seen_wndef)":   pos_seen ^ neg_seen,
}
strat_rows = []
for stratum, mask in strata.items():
    if mask.sum() == 0:
        continue
    a_rows = anchor_rows[mask]
    p_rows = pos_rows[mask]
    n_rows = neg_rows[mask]
    for model in MODEL_NAMES:
        A = embeddings[(model, "f_clue_val")][a_rows]
        P = embeddings[(model, "f_common_wndef")][p_rows]
        N = embeddings[(model, "f_common_wndef")][n_rows]
        stats = triplet_eval(A, P, N)
        strat_rows.append({
            "Stratum":        stratum,
            "Model":          model,
            "N":              stats["n"],
            "L2 accuracy":    stats["l2_accuracy"],
            "Cos accuracy":   stats["cos_accuracy"],
            "L2 mean margin": stats["l2_mean_margin"],
            "Cos mean margin":stats["cos_mean_margin"],
        })
strat_df = pd.DataFrame(strat_rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(strat_df.to_string(index=False))

In [ ]:
# === Figure: seen/unseen stratification (g1be_seen_unseen.png)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# Left: mean L2 norms by word set and model (grouped bars). L2 data -> //
# hatching per FIGURE_STANDARDS.md, even though the panel contains only L2.
ax = axes[0]
set_order = ["seen_wndef", "seen_fclue_only", "unseen"]
x = np.arange(len(set_order))
width = 0.38
for i, model in enumerate(MODEL_NAMES):
    vals = [mean_norms_by_set[(model, s)] for s in set_order]
    ax.bar(x + (i - 0.5) * width, vals, width=width,
           color=MODEL_COLORS[model], hatch="//",
           edgecolor="white", linewidth=0.5, label=model)
ax.set_xticks(x)
ax.set_xticklabels(set_order, rotation=15, ha="right")
ax.set_ylabel("Mean L2 norm (wndef embedding)")
ax.set_title("Embedding norms by word-set exposure")
ax.grid(alpha=0.3, axis="y")
ax.legend(loc="best")

# Right: triplet accuracy by stratum, L2 and cos side by side. Same panel,
# two metrics -> cosine solid, L2 "//" hatched per FIGURE_STANDARDS.md.
# 95% CI error bars on each bar (SE = sqrt(p*(1-p)/N), CI = +-1.96*SE).
ax = axes[1]
stratum_order = ["both seen_wndef", "mixed (one seen_wndef)", "both unseen"]
present_strata = [s for s in stratum_order if s in strat_df["Stratum"].unique()]
x = np.arange(len(present_strata))
bar_w = 0.2

plot_specs = [
    ("g_stock", "L2 accuracy",  MODEL_COLORS["g_stock"], "//"),
    ("g1",      "L2 accuracy",  MODEL_COLORS["g1"],      "//"),
    ("g_stock", "Cos accuracy", MODEL_COLORS["g_stock"], ""),
    ("g1",      "Cos accuracy", MODEL_COLORS["g1"],      ""),
]
for j, (model, metric, color, hatch) in enumerate(plot_specs):
    vals, errs = [], []
    for s in present_strata:
        row = strat_df[(strat_df["Stratum"] == s) & (strat_df["Model"] == model)]
        p = float(row[metric].iloc[0])
        n = int(row["N"].iloc[0])
        vals.append(p)
        errs.append(1.96 * np.sqrt(p * (1 - p) / n))
    ax.bar(x + (j - 1.5) * bar_w, vals, width=bar_w, color=color, hatch=hatch,
           yerr=errs, capsize=3, ecolor="black",
           error_kw={"elinewidth": 0.8},
           label=f"{model} / {metric}", edgecolor="white", linewidth=0.5)

ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, zorder=0,
           label="chance (0.5)")
ax.set_xticks(x)
ax.set_xticklabels(present_strata, rotation=15, ha="right")
ax.set_ylabel("Triplet accuracy")
ax.set_title("Triplet accuracy by positive/negative exposure")
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3, axis="y")
ax.legend(loc="lower right", fontsize=8, ncol=2)

fig.tight_layout()
fig.savefig(FIG_DIR / "g1be_seen_unseen.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# === Figure: cosine triplet accuracy by stratum (g1be_seen_unseen_cosine.png)
# Report-facing single panel. Cosine only, so solid fill (no hatching).
# Model identity uses MODEL_COLORS per FIGURE_STANDARDS.md. 95% CI error
# bars on each bar (SE = sqrt(p*(1-p)/N), CI = +-1.96*SE).
fig, ax = plt.subplots(figsize=(7.5, 4.8))

stratum_order = ["both seen_wndef", "mixed (one seen_wndef)", "both unseen"]
present_strata = [s for s in stratum_order if s in strat_df["Stratum"].unique()]
x = np.arange(len(present_strata))
bar_w = 0.36

for i, model in enumerate(MODEL_NAMES):
    vals, errs = [], []
    for s in present_strata:
        row = strat_df[(strat_df["Stratum"] == s) & (strat_df["Model"] == model)]
        p = float(row["Cos accuracy"].iloc[0])
        n = int(row["N"].iloc[0])
        vals.append(p)
        errs.append(1.96 * np.sqrt(p * (1 - p) / n))
    ax.bar(x + (i - 0.5) * bar_w, vals, width=bar_w,
           color=MODEL_COLORS[model], edgecolor="white", linewidth=0.5,
           yerr=errs, capsize=3, ecolor="black",
           error_kw={"elinewidth": 0.8}, label=model)

ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, zorder=0,
           label="chance (0.5)")
ax.set_xticks(x)
ax.set_xticklabels(present_strata, rotation=15, ha="right")
ax.set_ylabel("Cosine triplet accuracy")
ax.set_title("Cosine triplet accuracy by vocab exposure (Validation Set)")
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3, axis="y")
ax.legend(loc="lower right", fontsize=9)

fig.tight_layout()
fig.savefig(FIG_DIR / "g1be_seen_unseen_cosine.png", dpi=300, bbox_inches="tight")
plt.show()

### §4e — Compression summary

Deltas between g_stock and g1 aggregated across measures in this section.

In [ ]:
# === Compression summary deltas
def delta(stock_val, g1_val):
    return g1_val - stock_val

compression_rows = []
for phrase in ["f_common_wndef", "f_clue_val"]:
    pop_label = "wndef vocab" if phrase == "f_common_wndef" else "f_clue val"
    stock_mean_norm = float(norms[("g_stock", phrase)].mean())
    g1_mean_norm    = float(norms[("g1",      phrase)].mean())
    stock_pair_cos  = float(pair_sims[("g_stock", phrase)].mean())
    g1_pair_cos     = float(pair_sims[("g1",      phrase)].mean())
    _, stock_total, stock_eff, _ = eff_data[("g_stock", phrase)]
    _, g1_total,    g1_eff,    _ = eff_data[("g1",     phrase)]
    compression_rows.append({
        "Population":           pop_label,
        "Δ mean norm":          delta(stock_mean_norm, g1_mean_norm),
        "Δ mean pair cos":      delta(stock_pair_cos, g1_pair_cos),
        "Δ total variance":     delta(stock_total, g1_total),
        "% total variance":     (g1_total / stock_total - 1) * 100,
        "Δ effective dim":      delta(stock_eff, g1_eff),
    })
compression_df = pd.DataFrame(compression_rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(compression_df.to_string(index=False))

# Seen vs unseen differential: norm shrink for seen_wndef vs unseen.
seen_delta  = mean_norms_by_set[("g1", "seen_wndef")] - mean_norms_by_set[("g_stock", "seen_wndef")]
unseen_delta= mean_norms_by_set[("g1", "unseen")]     - mean_norms_by_set[("g_stock", "unseen")]
print(f"\nNorm Δ seen_wndef: {seen_delta:+.4f}")
print(f"Norm Δ unseen:     {unseen_delta:+.4f}")
print(f"Seen-vs-unseen norm differential: {seen_delta - unseen_delta:+.4f}")

## §5 — Structural comparison to g_stock

If two words were similar under `g_stock`, are they still similar under
`g1`? Spearman rank correlation between the upper triangles of the
pairwise-cosine matrices answers this directly. Near-zero rho means
fine-tuning fundamentally reorganized the similarity structure; high rho
means relative similarities were preserved (even if magnitudes changed).

In [ ]:
# === Spearman on pairwise-cosine upper triangles
N_SUBSAMPLE = 1000

def pairwise_upper_tri(emb, rows):
    M = emb[rows]
    M_norm = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-10)
    sim = M_norm @ M_norm.T
    i, j = np.triu_indices(len(rows), k=1)
    return sim[i, j]

spearman_rows = []
rng = np.random.default_rng(RANDOM_STATE)

# (a) wndef vocab: sample 1,000 rows from the 53,930-word vocabulary.
wndef_idx = rng.choice(embeddings[("g_stock", "f_common_wndef")].shape[0],
                       size=N_SUBSAMPLE, replace=False)
tri_stock_w = pairwise_upper_tri(embeddings[("g_stock", "f_common_wndef")], wndef_idx)
tri_g1_w    = pairwise_upper_tri(embeddings[("g1",      "f_common_wndef")], wndef_idx)
rho_w, p_w = spearmanr(tri_stock_w, tri_g1_w)
spearman_rows.append({
    "Population":   "wndef vocab",
    "N words":      N_SUBSAMPLE,
    "N pairs":      len(tri_stock_w),
    "Spearman rho": float(rho_w),
    "p-value":      float(p_w),
})

# (b) f_clue val: sample 1,000 rows from the 47,933 validation clues.
clue_idx = rng.choice(embeddings[("g_stock", "f_clue_val")].shape[0],
                      size=N_SUBSAMPLE, replace=False)
tri_stock_c = pairwise_upper_tri(embeddings[("g_stock", "f_clue_val")], clue_idx)
tri_g1_c    = pairwise_upper_tri(embeddings[("g1",      "f_clue_val")], clue_idx)
rho_c, p_c = spearmanr(tri_stock_c, tri_g1_c)
spearman_rows.append({
    "Population":   "f_clue val",
    "N words":      N_SUBSAMPLE,
    "N pairs":      len(tri_stock_c),
    "Spearman rho": float(rho_c),
    "p-value":      float(p_c),
})

spearman_df = pd.DataFrame(spearman_rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(spearman_df.to_string(index=False))

## §6 — Does g1 exploit cryptic clue structure?

Cryptic clues contain structure unique to the genre — surface
misdirection, wordplay pointing to the answer, letter-level tricks —
that a well-trained model might learn to exploit. The ATE measures
whether `g1` extracts useful information from clue context, without
distinguishing which mechanism (overcoming misdirection vs discovering
helpful cryptic structure) is operating.

### §6a — T=0 and T=1 similarity distributions (cosine)

For each of the 47,933 validation pairs we compare the decontextualized
definition→answer similarity (T=0) with the clue-contextualized
definition→answer similarity (T=1).

In [ ]:
# === Assemble eval pairs: (clue_row, def_row, ans_row) for 47,933 val rows
t0 = time.time()

eval_clue_rows = np.empty(len(clues_val), dtype=np.int64)
eval_def_rows  = np.empty(len(clues_val), dtype=np.int64)
eval_ans_rows  = np.empty(len(clues_val), dtype=np.int64)
n_miss = {"clue": 0, "def": 0, "ans": 0}

for i, (cid, orig_def, def_wn, ans_wn) in enumerate(zip(
    clues_val["clue_id"], clues_val["definition"],
    clues_val["definition_wn"], clues_val["answer_wn"],
)):
    eval_clue_rows[i] = clue_key_to_row.get((cid, orig_def), -1)
    eval_def_rows[i]  = wndef_word_to_row.get(def_wn, -1)
    eval_ans_rows[i]  = wndef_word_to_row.get(ans_wn, -1)
    if eval_clue_rows[i] < 0: n_miss["clue"] += 1
    if eval_def_rows[i]  < 0: n_miss["def"]  += 1
    if eval_ans_rows[i]  < 0: n_miss["ans"]  += 1

keep = (eval_clue_rows >= 0) & (eval_def_rows >= 0) & (eval_ans_rows >= 0)
eval_clue_rows = eval_clue_rows[keep]
eval_def_rows  = eval_def_rows[keep]
eval_ans_rows  = eval_ans_rows[keep]

print(f"clues_val rows:       {len(clues_val):,}")
print(f"  missing clue row:   {n_miss['clue']:,}")
print(f"  missing def row:    {n_miss['def']:,}")
print(f"  missing ans row:    {n_miss['ans']:,}")
print(f"  kept:               {len(eval_clue_rows):,} "
      f"({len(eval_clue_rows)/len(clues_val):.1%})")
print(f"Assemble: {time.time() - t0:.1f}s")

In [ ]:
# === T=0 and T=1 cosine similarities for both models
def t0_t1_cosine(model):
    clue_emb  = embeddings[(model, "f_clue_val")]
    vocab_emb = embeddings[(model, "f_common_wndef")]
    T0 = rowwise_cosine(vocab_emb[eval_def_rows], vocab_emb[eval_ans_rows])
    T1 = rowwise_cosine(clue_emb[eval_clue_rows], vocab_emb[eval_ans_rows])
    return T0, T1

cos_t01 = {model: t0_t1_cosine(model) for model in MODEL_NAMES}

def dist_stats(label, v):
    return {
        "Distribution": label,
        "Mean":   float(v.mean()),
        "Median": float(np.median(v)),
        "Std":    float(v.std()),
        "P5":     float(np.percentile(v, 5)),
        "P95":    float(np.percentile(v, 95)),
    }

cos_rows = []
for model in MODEL_NAMES:
    T0, T1 = cos_t01[model]
    cos_rows.append(dist_stats(f"{model} T=0", T0))
    cos_rows.append(dist_stats(f"{model} T=1", T1))
cos_dist_df = pd.DataFrame(cos_rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(cos_dist_df.to_string(index=False))

In [ ]:
# === ATE (cosine): mean(T=1 - T=0), SE, 95% CI, % negative
def ate_summary(T0, T1):
    delta = T1 - T0
    n = len(delta)
    mean = float(delta.mean())
    med  = float(np.median(delta))
    se   = float(delta.std(ddof=1) / np.sqrt(n))
    ci_lo = mean - 1.96 * se
    ci_hi = mean + 1.96 * se
    pct_neg = float((delta < 0).mean())
    return {
        "N":           n,
        "ATE (mean)":  mean,
        "Median Δ":    med,
        "SE":          se,
        "95% CI lo":   ci_lo,
        "95% CI hi":   ci_hi,
        "% Δ < 0":     pct_neg * 100,
    }

ate_cos_rows = []
for model in MODEL_NAMES:
    T0, T1 = cos_t01[model]
    rec = ate_summary(T0, T1)
    rec = {"Model": model, **rec}
    ate_cos_rows.append(rec)
ate_cos_df = pd.DataFrame(ate_cos_rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(ate_cos_df.to_string(index=False))

### §6b — T=0 and T=1 distances (L2)

For L2 distance, *smaller* = more similar (opposite of cosine). The L2
context effect is `T=1 distance − T=0 distance`; positive values mean
context pushes the definition farther from the answer.

In [ ]:
# === T=0 and T=1 L2 distances for both models
def t0_t1_l2(model):
    clue_emb  = embeddings[(model, "f_clue_val")]
    vocab_emb = embeddings[(model, "f_common_wndef")]
    D0 = np.linalg.norm(vocab_emb[eval_def_rows] - vocab_emb[eval_ans_rows], axis=1)
    D1 = np.linalg.norm(clue_emb[eval_clue_rows] - vocab_emb[eval_ans_rows], axis=1)
    return D0, D1

l2_t01 = {model: t0_t1_l2(model) for model in MODEL_NAMES}

l2_dist_rows = []
for model in MODEL_NAMES:
    D0, D1 = l2_t01[model]
    l2_dist_rows.append({
        "Metric":                     "T=0 L2 distance (mean)",
        "g_stock": float(l2_t01["g_stock"][0].mean()) if model == "g_stock" else None,
        "g1":      float(l2_t01["g1"][0].mean())      if model == "g1"      else None,
    })
# Build the table more simply.
l2_rows = []
for model in MODEL_NAMES:
    D0, D1 = l2_t01[model]
    delta = D1 - D0
    n = len(delta)
    se = float(delta.std(ddof=1) / np.sqrt(n))
    l2_rows.append({
        "Model":                              model,
        "T=0 L2 distance (mean)":             float(D0.mean()),
        "T=1 L2 distance (mean)":             float(D1.mean()),
        "L2 context effect (T=1 - T=0, mean)":float(delta.mean()),
        "SE":                                  se,
        "95% CI lo":                           float(delta.mean() - 1.96 * se),
        "95% CI hi":                           float(delta.mean() + 1.96 * se),
        "% pairs context increases distance": float((delta > 0).mean()) * 100,
    })
l2_df = pd.DataFrame(l2_rows).set_index("Model").T
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(l2_df.to_string())

### §6c — Figures

In [ ]:
# === Figure: T=0 / T=1 cosine histograms (g1be_t0_t1_cosine.png)
# Per FIGURE_STANDARDS.md, T=0 is outline-only (histtype="step") and T=1 is
# filled, both in the model's full hue. Cosine -> no hatching.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

all_cos = np.concatenate([cos_t01[m][k] for m in MODEL_NAMES for k in (0, 1)])
xmin, xmax = float(np.percentile(all_cos, 0.5)), float(np.percentile(all_cos, 99.5))
common_bins = np.linspace(xmin, xmax, 61)

for ax, model in zip(axes, MODEL_NAMES):
    T0, T1 = cos_t01[model]
    c = MODEL_COLORS[model]
    # Filled T=1 first so the T=0 outline sits on top and stays readable.
    ax.hist(T1, bins=common_bins, histtype="stepfilled", color=c,
            alpha=0.55, label="T=1 (filled)", density=True)
    ax.hist(T0, bins=common_bins, histtype="step", color=c,
            linewidth=1.8, label="T=0 (outline)", density=True)
    ax.axvline(T0.mean(), color=c, linestyle="--", linewidth=1)
    ax.axvline(T1.mean(), color=c, linestyle="-",  linewidth=1)
    ax.set_xlim(xmin, xmax)
    ax.set_xlabel("Cosine similarity")
    ax.set_ylabel("Density")
    ax.set_title(f"{model}: T=0 vs T=1 (cosine)")
    ax.grid(alpha=0.3)
    ax.legend(loc="best")

fig.tight_layout()
fig.savefig(FIG_DIR / "g1be_t0_t1_cosine.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# === Figure: T=0 / T=1 L2 histograms (g1be_t0_t1_l2.png)
# Same outline/fill scheme as g1be_t0_t1_cosine.png; L2 data -> T=1 filled
# histogram carries "//" diagonal hatching per FIGURE_STANDARDS.md.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

all_l2 = np.concatenate([l2_t01[m][k] for m in MODEL_NAMES for k in (0, 1)])
xmin, xmax = float(np.percentile(all_l2, 0.5)), float(np.percentile(all_l2, 99.5))
common_bins = np.linspace(xmin, xmax, 61)

for ax, model in zip(axes, MODEL_NAMES):
    D0, D1 = l2_t01[model]
    c = MODEL_COLORS[model]
    ax.hist(D1, bins=common_bins, histtype="stepfilled", color=c,
            alpha=0.55, hatch="//", edgecolor=c,
            label="T=1 (filled)", density=True)
    ax.hist(D0, bins=common_bins, histtype="step", color=c,
            linewidth=1.8, label="T=0 (outline)", density=True)
    ax.axvline(D0.mean(), color=c, linestyle="--", linewidth=1)
    ax.axvline(D1.mean(), color=c, linestyle="-",  linewidth=1)
    ax.set_xlim(xmin, xmax)
    ax.set_xlabel("L2 distance")
    ax.set_ylabel("Density")
    ax.set_title(f"{model}: T=0 vs T=1 (L2)")
    ax.grid(alpha=0.3)
    ax.legend(loc="best")

fig.tight_layout()
fig.savefig(FIG_DIR / "g1be_t0_t1_l2.png", dpi=300, bbox_inches="tight")
plt.show()

### §6d — Interpretation

The cosine ATE and the L2 context effect both summarize the "does
context help?" question, but they may tell different stories because the
two metrics weigh angular and magnitude structure differently. The cell
below prints side-by-side summaries so the comparison is explicit; the
written interpretation belongs in the results markdown and in the final
summary cell.

In [ ]:
# === Side-by-side cosine / L2 context effect summary
ce_rows = []
for model in MODEL_NAMES:
    T0_c, T1_c = cos_t01[model]
    D0,  D1    = l2_t01[model]
    ce_rows.append({
        "Model":                    model,
        "Cosine ATE (T=1 - T=0)":   float((T1_c - T0_c).mean()),
        "Cosine mean T=0":          float(T0_c.mean()),
        "Cosine mean T=1":          float(T1_c.mean()),
        "L2 context (T=1 - T=0)":   float((D1  - D0  ).mean()),
        "L2 mean T=0":              float(D0.mean()),
        "L2 mean T=1":              float(D1.mean()),
    })
context_df = pd.DataFrame(ce_rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(context_df.to_string(index=False))

## §7 — Write results file

All key numerical results are serialized to
`outputs/g1_basic_evaluation-results.md` so the Architect can review the
notebook's findings without re-executing it.

In [ ]:
# === Build outputs/g1_basic_evaluation-results.md
t0 = time.time()
RESULTS_PATH = OUTPUT_DIR / "results" / "g1_basic_evaluation-results.md"

def fmt_table(df, float_fmt="{:.4f}"):
    return df.to_markdown(index=False, floatfmt=float_fmt.format(0).replace("0", ""))

# Results-file helper: reuse the pandas float formatting through a context.
def df_md(df, fmt=".4f"):
    return df.to_markdown(index=False, floatfmt=fmt)

lines = []
lines.append("# g1 Basic Model Evaluation — Results")
lines.append("")
lines.append(f"Generated: {date.today().isoformat()}")
lines.append("")
lines.append("## Versions")
lines.append("")
for k, v in VERSIONS.items():
    lines.append(f"- **{k}:** {v}")
lines.append("")

lines.append("## §2 — Training dynamics")
lines.append("")
lines.append(df_md(td_df, fmt=".4f"))
lines.append("")
lines.append(f"Best-generalizing checkpoint: epoch {best_epoch}.  "
             f"Deployed checkpoint: epoch {deployed_epoch}.")
lines.append("")

lines.append("## §3 — Task performance")
lines.append("")
lines.append("### §3b — Validation triplet accuracy (wndef, full-vocab)")
lines.append("")
lines.append(df_md(val_triplet_df, fmt=".4f"))
lines.append("")
lines.append("### §3d — Summary table")
lines.append("")
lines.append(task_summary_df.to_markdown(index=False))
lines.append("")

lines.append("## §4 — Embedding space geometry")
lines.append("")
lines.append("### §4a — Norm distributions")
lines.append("")
lines.append(df_md(norm_df, fmt=".4f"))
lines.append("")
lines.append("### §4b — Pairwise cosine among random word pairs (N=50,000)")
lines.append("")
lines.append(df_md(pair_df, fmt=".4f"))
lines.append("")
lines.append("### §4c — Effective dimensionality")
lines.append("")
lines.append(df_md(eff_df, fmt=".2f"))
lines.append("")
lines.append("### §4d — Seen/unseen stratification")
lines.append("")
lines.append("Norms by word set:")
lines.append("")
lines.append(df_md(norm_strat_df, fmt=".4f"))
lines.append("")
lines.append("Pairwise cosine (seen_wndef vs unseen):")
lines.append("")
lines.append(df_md(pair_strat_df, fmt=".4f"))
lines.append("")
lines.append("Triplet accuracy stratified by pos/neg exposure:")
lines.append("")
lines.append(df_md(strat_df, fmt=".4f"))
lines.append("")
lines.append("### §4e — Compression summary")
lines.append("")
lines.append(df_md(compression_df, fmt=".4f"))
lines.append("")

lines.append("## §5 — Structural comparison to g_stock")
lines.append("")
lines.append(df_md(spearman_df, fmt=".4g"))
lines.append("")

lines.append("## §6 — Context effects")
lines.append("")
lines.append("### §6a — Cosine T=0 / T=1 distributions")
lines.append("")
lines.append(df_md(cos_dist_df, fmt=".4f"))
lines.append("")
lines.append("### §6a — Cosine ATE")
lines.append("")
lines.append(df_md(ate_cos_df, fmt=".4f"))
lines.append("")
lines.append("### §6b — L2 T=0 / T=1 and context effect")
lines.append("")
lines.append(l2_df.to_markdown())
lines.append("")
lines.append("### §6d — Cosine vs L2 context effect summary")
lines.append("")
lines.append(df_md(context_df, fmt=".4f"))
lines.append("")

lines.append("## Figures")
lines.append("")
for fig_name in [
    "g1be_training_dynamics.png",
    "g1be_norm_distributions.png",
    "g1be_pairwise_cosine.png",
    "g1be_singular_values.png",
    "g1be_seen_unseen.png",
    "g1be_seen_unseen_cosine.png",
    "g1be_t0_t1_cosine.png",
    "g1be_t0_t1_l2.png",
]:
    lines.append(f"- `figures/{fig_name}`")
lines.append("")

RESULTS_PATH.write_text("\n".join(lines))
print(f"Wrote {RESULTS_PATH} ({RESULTS_PATH.stat().st_size:,} bytes) "
      f"in {time.time() - t0:.1f}s")

## Summary

### Key findings

- **§2 — Training dynamics:** validation loss reached its minimum at
  epoch 2 (0.252) and rose slightly at epoch 3 (0.264), while training
  loss continued dropping to 0.014. The deployed epoch-3 checkpoint
  therefore sits past the generalization optimum; the val/train loss
  ratio at epoch 3 is ~19×, a clear overfitting signal.
- **§3 — Task performance:** validation triplet accuracy is reported
  under both L2 (training metric) and cosine (research metric). Any gap
  between the two quantifies how much of g1's learning is magnitude-
  versus angle-based.
- **§4 — Embedding space geometry:** norm, pairwise-cosine,
  effective-dimensionality, and seen/unseen-stratified measures
  together characterize whether fine-tuning compressed, crowded, or
  selectively reshaped the embedding space.
- **§5 — Structural comparison:** Spearman rho on pairwise-cosine upper
  triangles indicates how much of g_stock's similarity structure
  survived fine-tuning.
- **§6 — Context effects:** the cosine ATE and the L2 context effect
  separately summarize whether clue context helps the model connect
  definition to answer, using the metric the model was trained on
  alongside the metric the project uses for research.

### Outputs written

- `outputs/g1_basic_evaluation-results.md`
- `outputs/figures/g1be_training_dynamics.png`
- `outputs/figures/g1be_norm_distributions.png`
- `outputs/figures/g1be_pairwise_cosine.png`
- `outputs/figures/g1be_singular_values.png`
- `outputs/figures/g1be_seen_unseen.png`
- `outputs/figures/g1be_seen_unseen_cosine.png`
- `outputs/figures/g1be_t0_t1_cosine.png`
- `outputs/figures/g1be_t0_t1_l2.png`

### Wish-list and next steps

- We could not compute training triplet accuracy directly because `g1`
  predates Decision 25 (which requires `f_clue_train` embeddings for
  every fine-tuned model). It is bounded below by the hinge-loss
  identity at epoch 3; a precise number awaits g2 or a retrofit.
- Based on §4's crowding / compression findings and §6's context
  effect, the notebook forms a testable prediction about where g1
  would rank the true answer relative to g_stock in a retrieval
  setting — that prediction, stated in §6d, should be validated in a
  dedicated retrieval analysis.
- Findings here motivate g2 design under Decisions 25 and 26:
  training-split f_clue embeddings plus both-metric evaluation should
  be built in from the start.

In [ ]:
# === Wall-clock runtime
elapsed = time.time() - NOTEBOOK_T0
print(f"Notebook wall-clock runtime: {elapsed:.1f}s ({elapsed/60:.1f} min)")